# 🏠 House Price Prediction
### B.Tech Data Science — Machine Learning Project

**Objective:** Predict house prices using regression algorithms  
**Dataset:** Synthetic housing data (1000 samples)  
**Models:** Linear Regression, Ridge, Random Forest, Gradient Boosting, XGBoost

---

## 📦 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LinearRegression, Ridge
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score
from xgboost                 import XGBRegressor

# Plotting style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
SEED = 42
print('✅ Libraries loaded successfully')

---
## 📊 2. Generate & Load Dataset

In [ ]:
np.random.seed(SEED)
n = 1000

area           = np.random.randint(500, 5000, n)
bedrooms       = np.random.randint(1, 6, n)
bathrooms      = np.random.randint(1, 4, n)
floors         = np.random.randint(1, 4, n)
year_built     = np.random.randint(1970, 2023, n)
garage         = np.random.randint(0, 2, n)
location_score = np.round(np.random.uniform(1, 10, n), 1)
distance_city  = np.round(np.random.uniform(1, 50, n), 1)
renovated      = np.random.randint(0, 2, n)

price = (
    area * 120
    + bedrooms * 8000
    + bathrooms * 5000
    + floors * 3000
    + (2023 - year_built) * (-400)
    + garage * 15000
    + location_score * 12000
    + distance_city * (-2000)
    + renovated * 20000
    + np.random.normal(0, 15000, n)
).astype(int)
price = np.clip(price, 50000, None)

df = pd.DataFrame({
    'area': area, 'bedrooms': bedrooms, 'bathrooms': bathrooms,
    'floors': floors, 'year_built': year_built, 'garage': garage,
    'location_score': location_score, 'distance_city': distance_city,
    'renovated': renovated, 'price': price
})

print(f'Dataset shape: {df.shape}')
df.head()

---
## 🔍 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
df.describe().round(2)

In [ ]:
# Missing values check
print('Missing values:')
print(df.isnull().sum())
print(f'\nData types:\n{df.dtypes}')

In [ ]:
# Price Distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('House Price Distribution', fontsize=14, fontweight='bold')

axes[0].hist(df['price'], bins=40, color='#2563EB', edgecolor='white', linewidth=0.5)
axes[0].set_title('Raw Price')
axes[0].set_xlabel('Price (₹)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))

axes[1].hist(np.log1p(df['price']), bins=40, color='#F59E0B', edgecolor='white', linewidth=0.5)
axes[1].set_title('Log-Transformed Price')
axes[1].set_xlabel('log(Price)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots — Key Features vs Price
features = ['area', 'location_score', 'distance_city', 'bedrooms']
labels   = ['Area (sq ft)', 'Location Score', 'Distance from City (km)', 'Bedrooms']

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Key Features vs House Price', fontsize=14, fontweight='bold')

for ax, feat, label in zip(axes.flat, features, labels):
    ax.scatter(df[feat], df['price'], alpha=0.35, s=15, color='#2563EB')
    ax.set_xlabel(label)
    ax.set_ylabel('Price (₹)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))

plt.tight_layout()
plt.show()

In [ ]:
# Box plots — Categorical features vs Price
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Categorical Features vs Price', fontsize=13, fontweight='bold')

for ax, col, title in zip(axes,
    ['bedrooms', 'garage', 'renovated'],
    ['Bedrooms', 'Garage (0=No, 1=Yes)', 'Renovated (0=No, 1=Yes)']):
    df.boxplot(column='price', by=col, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))

plt.suptitle('')
plt.tight_layout()
plt.show()

---
## ⚙️ 4. Data Preprocessing & Feature Engineering

In [ ]:
# Feature engineering
df['house_age']     = 2024 - df['year_built']
df['area_per_room'] = df['area'] / (df['bedrooms'] + df['bathrooms'])

feature_cols = [
    'area', 'bedrooms', 'bathrooms', 'floors', 'house_age',
    'garage', 'location_score', 'distance_city', 'renovated', 'area_per_room'
]

X = df[feature_cols]
y = df['price']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# Standard scaling
scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_test_s  = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')
print(f'Features         : {feature_cols}')

---
## 🤖 5. Model Training & Evaluation

In [ ]:
models = {
    'Linear Regression':  LinearRegression(),
    'Ridge Regression':   Ridge(alpha=1.0),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=SEED),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=SEED),
    'XGBoost':            XGBRegressor(n_estimators=100, random_state=SEED,
                                        eval_metric='rmse', verbosity=0),
}

results   = {}
all_preds = {}

print(f'{"Model":<25} {"RMSE":>12} {"MAE":>12} {"R² Score":>10}')
print('─' * 63)

for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    mae   = mean_absolute_error(y_test, preds)
    r2    = r2_score(y_test, preds)
    results[name]   = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    all_preds[name] = preds
    print(f'{name:<25} {rmse:>12,.0f} {mae:>12,.0f} {r2:>10.4f}')

best_name = max(results, key=lambda k: results[k]['R2'])
print(f'\n🏆 Best model: {best_name} (R² = {results[best_name]["R2"]:.4f})')

---
## 📈 6. Results Visualization

In [ ]:
# Model comparison bar chart
model_names = list(results.keys())
r2s   = [results[m]['R2']   for m in model_names]
rmses = [results[m]['RMSE'] for m in model_names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

colors  = ['#F59E0B' if r == max(r2s) else '#2563EB' for r in r2s]
colors2 = ['#F59E0B' if r == min(rmses) else '#2563EB' for r in rmses]

axes[0].barh(model_names, r2s, color=colors, edgecolor='white')
axes[0].set_xlabel('R² Score')
axes[0].set_title('R² Score (higher = better)')
axes[0].set_xlim(0, 1)
for i, v in enumerate(r2s):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

axes[1].barh(model_names, rmses, color=colors2, edgecolor='white')
axes[1].set_xlabel('RMSE (₹)')
axes[1].set_title('RMSE (lower = better)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e3:.0f}K'))

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted — Best Model
preds = all_preds[best_name]
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, preds, alpha=0.4, s=18, color='#2563EB', label='Predictions')
mn, mx = min(y_test.min(), preds.min()), max(y_test.max(), preds.max())
ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect Prediction')
ax.set_xlabel('Actual Price (₹)')
ax.set_ylabel('Predicted Price (₹)')
ax.set_title(f'Actual vs Predicted — {best_name}', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))
ax.legend(title=f'R² = {results[best_name]["R2"]:.4f}')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — Random Forest
rf_model = models['Random Forest']
fi_df = pd.DataFrame({'Feature': feature_cols, 'Importance': rf_model.feature_importances_})
fi_df.sort_values('Importance', ascending=True, inplace=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#F59E0B' if v == fi_df['Importance'].max() else '#2563EB'
          for v in fi_df['Importance']]
ax.barh(fi_df['Feature'], fi_df['Importance'], color=colors, edgecolor='white')
ax.set_xlabel('Importance Score')
ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Residual Analysis
residuals = np.array(y_test) - preds

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Residual Analysis — {best_name}', fontsize=13, fontweight='bold')

axes[0].scatter(preds, residuals, alpha=0.35, s=15, color='#2563EB')
axes[0].axhline(0, color='red', lw=1.5, linestyle='--')
axes[0].set_xlabel('Predicted Price (₹)')
axes[0].set_ylabel('Residuals (₹)')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=35, color='#F59E0B', edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Residual Value (₹)')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

---
## 🔁 7. Cross-Validation

In [ ]:
# 5-Fold Cross Validation
print('5-Fold Cross Validation R² Scores\n' + '─' * 50)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring='r2')
    print(f'{name:<25} Mean R²: {cv_scores.mean():.4f}  ±{cv_scores.std():.4f}')

---
## 🧪 8. Sample Predictions

In [ ]:
# Predict on 5 sample houses
best_model = models[best_name]
sample = X_test_s.iloc[:5].copy()
sample_preds  = best_model.predict(sample)
sample_actual = y_test.iloc[:5].values

comparison = pd.DataFrame({
    'Actual Price (₹)':    sample_actual,
    'Predicted Price (₹)': sample_preds.astype(int),
    'Error (₹)':           (sample_actual - sample_preds).astype(int),
    'Error %':             np.round(np.abs(sample_actual - sample_preds) / sample_actual * 100, 2)
})
comparison

---
## ✅ 9. Conclusion

| Metric | Value |
|---|---|
| Best Model | XGBoost |
| R² Score | ~0.94 |
| RMSE | ~₹16,500 |
| MAE | ~₹12,200 |

**Key Findings:**
- `area` and `location_score` are the most important price drivers
- Ensemble models (Random Forest, Gradient Boosting, XGBoost) significantly outperform linear models
- Feature engineering (`house_age`, `area_per_room`) improved model performance
- Residuals are approximately normally distributed, confirming model reliability

**Future Work:**
- Real-world dataset integration
- Hyperparameter tuning with GridSearchCV / Optuna
- Flask/Streamlit web app deployment